# <center>Project Algorithms and Combinatorial Optimization<br/><br/> Deliverable 2</center>

<br/>

**Group 1:**<br/>
**&nbsp;&nbsp;&nbsp;&nbsp;Benjamin Auger**<br/>
**&nbsp;&nbsp;&nbsp;&nbsp;Jessica Stephen**<br/>
**&nbsp;&nbsp;&nbsp;&nbsp;Amaury Cattoz**<br/>
**&nbsp;&nbsp;&nbsp;&nbsp;Tiago Rotta--Bori**<br/>
**&nbsp;&nbsp;&nbsp;&nbsp;Virgil Gayte**


<div style="page-break-after: always;"></div>

## 1. Problem Identification

### 1.1 Problem Context

The French Agency for Ecological Transition (ADEME) seeks to optimize last-mile delivery operations in urban French municipalities while maintaining service quality and minimizing environmental impact. The operational challenge lies in designing efficient routing plans for vehicles departing from a logistics depot, visiting partner municipalities, and returning to the depot, with each location served exactly once.

### 1.2 Problem Scope and Reformulation

The problem is reformulated as a **Vehicle Routing Problem (VRP)** with the following simplifications to align with project constraints:

- **Single vehicle:** One truck departs from a fixed depot, visits a set of customer locations, and returns to the depot.
- **Fixed distances:** Travel times/distances between locations are deterministic and fixed, derived from shortest-path computations on urban road networks.
- **Single depot:** All routes originate from and terminate at a unique, fixed distribution center.
- **Complete service:** Each customer location must be visited exactly once.

### 1.3 Alignment with Standard Problems

This formulation is a direct specialization of the **Traveling Salesman Problem (TSP)**, which is a fundamental NP-hard combinatorial optimization problem. The TSP seeks a Hamiltonian cycle through a weighted graph that minimizes total cost (distance or time). By restricting to a single vehicle with a fixed depot, the ADEME last-mile delivery challenge becomes equivalent to finding an optimal TSP tour originating and terminating at the depot.

## 2. Mathematical Definition of the Problem

### 2.1 Graph Formulation

Let $G=(V,A)$ be a directed graph with $V=\{0,1,\dots,n\}$ (depot $0$ and $n$ customers) and $A\subseteq V\times V\setminus\{(i,i)\}$ the feasible arcs. Each arc $(i,j)$ has a nonnegative travel cost $c_{ij}$ (shortest travel‑time path); costs may be asymmetric. Decision variables $x_{ij}\in\{0,1\}$ indicate whether arc $(i,j)$ is used, and auxiliary order variables $u_i$ (Miller–Tucker–Zemlin) enforce subtour elimination.


$$
\begin{aligned}
\min\quad & Z=\sum_{i\in V}\sum_{\substack{j\in V\\ j\neq i}} c_{ij}\,x_{ij} \\[6pt]
\text{s.t.}\quad &
\sum_{\substack{j\in V\\ j\neq i}} x_{ij} = 1 && \forall\, i\in V \quad\text{(outdegree = 1)}\\[4pt]
&
\sum_{\substack{i\in V\\ i\neq j}} x_{ij} = 1 && \forall\, j\in V \quad\text{(indegree = 1)}\\[4pt]
&
u_i - u_j + n\,x_{ij} \le n-1 && \forall\, i,j\in V\setminus\{0\},\ i\neq j \\[4pt]
&
1 \le u_i \le n-1 && \forall\, i\in V\setminus\{0\}\\[4pt]
&
x_{ij} \in \{0,1\} && \forall\, i,j\in V,\ i\neq j
\end{aligned}
$$

**Constraints:**

- Each customer location is visited exactly once.
- The vehicle starts and ends at the depot (vertex 0).
- The tour forms a single, closed cycle (Hamiltonian cycle).
- All edges in the tour must exist in the graph (i.e., all locations are reachable).

### 2.2 Example Instance

Input data are stored as:


- **City set:** list of identifiers for the depot and customer locations.
- **Cost matrix $C$:** symmetric $(n+1) \times (n+1)$ matrix whose entry $c_{ij}$ equals the travel time between cities $i$ and $j$.
- **Feasibility mask:** optional binary matrix marking forbidden arcs when certain roads are unavailable.


Decision variables $x_{ij}$ naturally form a binary adjacency matrix describing the selected tour, while the outputs include the ordered list of serviced cities, the total travel time, and any dual information provided by the solver.


A stylized dataset is shown below for a depot (0) and three cities (1, 2, 3):


| i \ j | 0 | 1 | 2 | 3 |
|-------|---|---|---|---|
| 0     | 0 | 14| 11| 16|
| 1     |14 | 0 | 9 | 10|
| 2     |11 | 9 | 0 | 7 |
| 3     |16 |10 | 7 | 0 |



All values are expressed in minutes. The optimization engine takes this matrix, enforces the MTZ constraints, and returns the best cycle that covers all cities exactly once.

## 3. Complexity Analysis

### 3.1 Computational Complexity of TSP

#### NP-Hardness

The **decision version of TSP** asks: "Does a tour of total cost at most $k$ exist?" This problem is **NP-complete**, as demonstrated by Karp (1972). The proof proceeds by polynomial reduction from the **Hamiltonian Cycle problem**, which is known to be NP-complete.

**Reduction:** Any instance of the Hamiltonian Cycle problem (where the question is whether a cycle visiting all vertices exactly once exists) can be transformed into a TSP instance by assigning weight 1 to edges present in the original graph and weight 2 to all other edges. A Hamiltonian cycle exists if and only if a TSP tour of cost $n$ (where $n$ is the number of vertices) exists in the weighted graph. This transformation is polynomial in time.

#### Implications for Algorithmic Approaches

The NP-hardness of TSP implies:

1. **No known polynomial-time algorithm exists** to solve arbitrary TSP instances optimally, unless $P = NP$ (widely believed to be false).
2. **Worst-case solution time grows exponentially** with the number of nodes.
3. **Exact algorithms** (branch-and-bound, branch-and-cut, dynamic programming) are computationally expensive for large instances.
4. **Heuristic and metaheuristic approaches** are practical alternatives, trading optimality guarantees for computational tractability.

### 3.2 Problem Complexity Relative to VRP Extensions

The current single-vehicle, single-depot formulation represents a **simplified variant** of the full Vehicle Routing Problem. Extensions such as multiple vehicles, time-window constraints, or vehicle capacity constraints increase problem complexity and remain NP-hard. The TSP studied here provides a foundation for understanding and addressing more complex routing challenges.

## 4. Justification of the Genetic Algorithm Approach

Given the NP-hardness of TSP established in Section 3, the selection of a **Genetic Algorithm** is motivated by the fundamental computational constraints and practical deployment requirements of this project.

### 4.1 Why Not Exact Algorithms?

**Exact Methods (Branch-and-Bound, Dynamic Programming)**

While exact algorithms guarantee optimal solutions, they are computationally prohibitive for realistic problem sizes:

- **Temporal Complexity:** The Held-Karp algorithm (Held & Karp 1962) solves TSP in $O(n^2 2^n)$ time. For $n = 20$ cities, this is $20^2 \cdot 2^{20} \approx 500$ million operations—already challenging. For $n = 50$, the computation is effectively infeasible.
- **Scalability:** ADEME's delivery network may encompass dozens or hundreds of customer locations. Exact methods cannot reliably solve such instances within operational time constraints (e.g., routing decisions needed within minutes or hours).
- **Memory Requirements:** Dynamic programming approaches require tabulation of subsets and partial solutions, consuming exponential memory.

### 4.2 Why Genetic Algorithms?

**Empirical Polynomial Tractability**

Genetic algorithms achieve near-optimal solutions in empirical polynomial time for realistic problem instances. A well-tuned GA can solve TSP instances with hundreds of cities in seconds to minutes, a capability far exceeding exact methods.

**Population-Based Search**

Unlike local search methods (2-Opt, 3-Opt) which may converge to local optima, GA maintains a diverse population of candidate solutions. This population structure reduces the risk of premature convergence and enables exploration of multiple regions of the solution space.

**Flexibility and Extensibility**

The GA framework naturally extends to incorporate additional constraints (time windows, capacity limits, multi-vehicle fleets). Modifications to chromosome representation or genetic operators can be made incrementally without fundamental algorithm redesign. In contrast, extending exact algorithms often requires substantial mathematical reworking.

**Solution Quality**

Empirical studies (Davis 1991; Whitley et al. 1989; Potvin 1996) demonstrate that genetic algorithms consistently achieve solutions within 2-5% of known optima on standard TSP benchmarks. This quality-to-computation tradeoff is attractive for operational scenarios where very minor suboptimality is acceptable in exchange for rapid solution deployment.

**Hybrid Capability**

GA can be combined with local search operators (2-Opt, 3-Opt) to refine solutions further. Such **memetic algorithms** or **hybrid metaheuristics** (GA + local search) achieve performance comparable to or exceeding pure GA, while remaining computationally practical.

## 5. Selected Solving Method: Genetic Algorithm

### 5.1 Justification and Rationale

A **Genetic Algorithm (GA)** is selected as the primary solution method for this project. This choice reflects the following considerations:

**Computational Tractability:**  
Given the NP-hardness of TSP, exact algorithms (branch-and-bound, dynamic programming) become computationally prohibitive as the number of customer locations grows beyond approximately 20-30 nodes. Genetic algorithms offer a practical balance, delivering near-optimal solutions in polynomial empirical time.

**Adaptability to Problem Variants:**  
Genetic algorithms naturally accommodate problem extensions such as time windows, vehicle capacity constraints, or multi-vehicle routing. The chromosome representation and genetic operators can be modified modularly without fundamental algorithmic restructuring.

**Implementation Maturity:**  
Genetic algorithms have been extensively studied for TSP and VRP contexts (Davis 1991; Whitley et al. 1989; Potvin 1996). Best practices for parameter tuning, crossover operators, and mutation strategies are well-established, reducing development risk.

**Integration with Project Objectives:**  
The GA framework aligns with the "Advanced Algorithms" work package focus. Implementing a metaheuristic from first principles demonstrates algorithmic design, population-based search, and fitness-driven optimization—core competencies in computer science education.

### 5.2 Algorithm Overview

A genetic algorithm evolves a population of candidate solutions (tours) over successive generations, applying selection, crossover, and mutation operators inspired by natural evolution.

**Population Representation:**  
Each solution is encoded as a **permutation chromosome**, where the $i$-th position represents the $i$-th customer visited in the tour. For example, the chromosome $[1, 3, 2, 0]$ represents the tour $0 \to 1 \to 3 \to 2 \to 0$.

**Fitness Evaluation:**  
The fitness of a chromosome is defined as the reciprocal of tour cost: $f(\text{tour}) = \frac{1}{\text{total\_cost}}$. Higher fitness values correspond to shorter tours, guiding the search toward optimal solutions.

**Selection:**  
Tournament selection or fitness-proportionate selection identifies promising individuals for reproduction. Selected parents undergo crossover to produce offspring, inheriting genetic material from both parents.

**Crossover Operators:**  
Standard crossover methods for permutation problems include:
- **Order Crossover (OX):** Preserves relative ordering of tour segments from each parent.
- **Partially Mapped Crossover (PMX):** Maps genes between parents while maintaining permutation validity.

**Mutation:**  
Mutation operators introduce variation by modifying individual chromosomes:
- **2-Opt Swap:** Reverses a segment of the permutation, equivalent to removing two edges and reconnecting.
- **Insertion Mutation:** Removes a customer from one position and reinserts at another.

**Termination Criteria:**  
The algorithm terminates upon reaching a maximum generation count, achieving a fitness threshold, or observing stagnation (e.g., no fitness improvement over $k$ consecutive generations).

### 5.3 Design Choices and Adaptations

**Chromosome Representation:**  
Permutation encoding is selected over binary encoding (representing each edge or arc as a binary bit) because permutation chromosomes directly represent valid tours and simplify constraint satisfaction. Invalid tours (e.g., those skipping customers or violating the single-vehicle assumption) are automatically excluded by the permutation structure.

**Fitness Function:**  
Tour cost minimization is encoded as fitness maximization via the reciprocal function. This transformation allows standard GA selection operators to favor shorter tours. Alternatively, fitness can be defined as $f(\text{tour}) = C_{\max} - \text{cost}(\text{tour})$, where $C_{\max}$ is a reference cost ensuring positive fitness values.

**Population Management:**  
The GA maintains a **steady-state population** of fixed size $N$. In each generation, two parent individuals are selected, crossed over and mutated to produce offspring, which then compete for survival. The worst individual(s) in the population may be replaced if offspring exhibit superior fitness.

**Elitism:**  
The best individual discovered so far is preserved across generations. This strategy prevents loss of high-quality solutions and ensures monotonic improvement in best fitness over time.

**Initialization:**  
The initial population is generated using a mix of strategies:
- **Random permutations:** Ensures population diversity.
- **Greedy nearest-neighbor heuristic:** Seeds the population with a reasonably good initial solution, accelerating convergence.

**Parameter Tuning:**  
Key parameters include population size, mutation probability, crossover probability, and termination criteria. These parameters are typically tuned empirically via experiments on benchmark instances or through sensitivity analysis.

## 6. Problem Constraints and Assumptions

### 6.1 Explicit Constraints (Project Scope)

**Single Vehicle:**  
The model assumes a single truck or delivery vehicle dispatched from the depot. Multi-vehicle variants would require distinct modifications to the chromosome representation (e.g., incorporating vehicle boundaries in the permutation) and fitness evaluation.

**Fixed Depot:**  
All routes originate and terminate at a unique, predetermined depot location (node 0). The depot is not a customer and incurs no delivery time or service requirement.

**Fixed Distances:**  
Arc costs $c_{ij}$ are deterministic and known a priori, derived from shortest-path computations or historical travel-time data. No stochastic travel time variability or dynamic congestion is modeled in this deliverable.

### 6.2 Implicit Assumptions

**Complete Graph:**  
The input graph is assumed to be complete, i.e., every customer is reachable from every other customer and from the depot. If certain roads are unavailable (e.g., due to closures), the cost matrix must reflect this by assigning prohibitively high costs or explicitly omitting arcs.

**Metric Property (Optional):**  
If distances represent shortest-path travel times in a road network, the cost matrix typically satisfies the metric property: $c_{ij} + c_{jk} \ge c_{ik}$ for all $i, j, k$. This property, while often present in real-world instances, is not enforced algorithmically and does not affect the GA's validity.

**Service at Single Location:**  
Each customer requires a single visit at a single location. Scenarios involving multiple service points or split deliveries are not addressed.

**No Time Windows:**  
In this deliverable, customer locations have no time-window constraints (e.g., "delivery must occur between 9:00 and 17:00"). Future extensions may incorporate temporal constraints via penalty functions or modified representations.

## 7. Implementation

This section details the implementation of the Genetic Algorithm metaheuristic for the Vehicle Routing Problem. The implementation consists of four core components that work together to execute the evolutionary computation process: population management, genetic operators, solution evaluation, and comparative benchmarking.

### 7.1 Architecture Overview

The implementation is organized into four functional components:

1. **Genetic Algorithm Core**: Implements the complete GA metaheuristic including population initialization, selection mechanisms, crossover operations, mutation procedures, and generational evolution
2. **VRP Exact Solver Interface**: Provides integration with state-of-the-art exact and heuristic solving methods for baseline comparison and validation
3. **Solution Comparison Framework**: Implements verification routines and comparative analysis to benchmark GA solutions against reference solutions
4. **Statistical Analysis Module**: Executes multiple independent GA runs and provides statistical visualization of performance distribution

### 7.2 Genetic Algorithm Core Implementation

The core GA implementation provides the complete evolutionary computation framework through the following key components:

#### Chromosome Representation and Population Management

Solutions are represented as permutations of customer indices, where each permutation defines a unique route visiting all customers. The population consists of multiple candidate solutions evolved through successive generations.

**Initialization**: The initial population is generated by creating random permutations of customer indices. This stochastic initialization ensures diverse starting solutions that explore different regions of the solution space.

#### Selection Mechanism

**Tournament Selection** is employed to select parent solutions for reproduction. This mechanism randomly selects a subset of individuals from the population (tournament size parameter) and chooses the individual with the best fitness value. This approach maintains selection pressure while preserving population diversity.

#### Genetic Operators

**Order Crossover (OX)**: This crossover operator combines two parent permutations while preserving the relative order of customers. The operation works by selecting a random segment from one parent, transferring it to the offspring, and filling remaining positions with customers from the second parent in their original order. This preserves the tour structure inherited from both parents.

**Swap Mutation**: Introduces diversity through controlled random modifications. With a given mutation probability, pairs of customers are randomly selected and their positions are exchanged within the permutation. This maintains the tour's validity (all customers visited exactly once) while exploring neighboring solutions.

#### Fitness Evaluation

Fitness is determined by calculating the total distance or cost of the route represented by each permutation. The fitness function computes the sum of distances between consecutive customers and returns to the depot. Lower costs indicate better fitness in this minimization problem.

#### Elitism and Termination

**Elitism** preserves the best-performing solutions across generations by copying elite individuals directly to the next generation without modification. This ensures that solution quality never degrades as the algorithm progresses.

**Adaptive Termination** halts evolution when no improvement is observed for a configurable number of generations (stagnation limit). This prevents unnecessary computation while maintaining solution quality.

In [1]:
"""
Genetic Algorithm (GA) Metaheuristic for VRP.

This module implements the Genetic Algorithm metaheuristic for solving Vehicle Routing Problems.
GA consists of the following main components:
1. Population Initialization: Create initial population of solutions
2. Selection: Select parents for reproduction
3. Crossover: Combine parents to create offspring
4. Mutation: Apply random changes to maintain diversity
5. Replacement: Form new generation from parents and offspring
"""

import numpy as np
import random
from typing import List, Tuple, Set, Optional
from pathlib import Path
import matplotlib.pyplot as plt
from copy import deepcopy


class GeneticAlgorithmSolver:
    """Genetic Algorithm metaheuristic solver for Vehicle Routing Problems."""
    
    def __init__(self, coords: np.ndarray, demands: np.ndarray, capacity: int, depot_idx: int = 0):
        """
        Initialize Genetic Algorithm solver.
        
        Args:
            coords: Array of (x, y) coordinates for all locations
            demands: Array of demands for each customer
            capacity: Vehicle capacity
            depot_idx: Index of the depot (default: 0)
        """
        self.coords = np.array(coords)
        self.demands = np.array(demands)
        self.capacity = capacity
        self.depot_idx = depot_idx
        self.num_locations = len(coords)
        self.distance_matrix = self._calculate_distance_matrix()
        
        self.best_solution = None
        self.best_cost = float('inf')
        self.generation_costs = []
        self.avg_costs = []
        
    def _calculate_distance_matrix(self) -> np.ndarray:
        """Calculate Euclidean distance matrix between all locations."""
        n = len(self.coords)
        distances = np.zeros((n, n))
        
        for i in range(n):
            for j in range(n):
                if i != j:
                    distances[i, j] = np.sqrt(
                        (self.coords[i][0] - self.coords[j][0]) ** 2 +
                        (self.coords[i][1] - self.coords[j][1]) ** 2
                    )
        
        return distances
    
    def _calculate_route_cost(self, route: List[int]) -> float:
        """Calculate the total cost of a route including return to depot."""
        if not route:
            return 0.0
        
        cost = self.distance_matrix[self.depot_idx, route[0]]  # Depot to first customer
        
        for i in range(len(route) - 1):
            cost += self.distance_matrix[route[i], route[i + 1]]
        
        cost += self.distance_matrix[route[-1], self.depot_idx]  # Last customer to depot
        
        return cost
    
    def _calculate_total_cost(self, routes: List[List[int]]) -> float:
        """Calculate total cost of all routes."""
        return sum(self._calculate_route_cost(route) for route in routes)
    
    def _is_feasible_route(self, route: List[int]) -> bool:
        """Check if a route respects capacity constraint."""
        total_demand = sum(self.demands[customer] for customer in route)
        return total_demand <= self.capacity
    
    def _convert_to_routes(self, chromosome: List[int]) -> List[List[int]]:
        """
        Convert a chromosome (permutation of customers) to routes.
        Split by capacity constraint.
        
        Args:
            chromosome: Permutation of customer indices
            
        Returns:
            List of routes
        """
        routes = []
        current_route = []
        current_capacity = 0
        
        for customer in chromosome:
            customer_demand = self.demands[customer]
            
            if current_capacity + customer_demand <= self.capacity:
                current_route.append(customer)
                current_capacity += customer_demand
            else:
                if current_route:
                    routes.append(current_route)
                current_route = [customer]
                current_capacity = customer_demand
        
        if current_route:
            routes.append(current_route)
        
        return routes
    
    def _evaluate_fitness(self, chromosome: List[int]) -> float:
        """
        Evaluate fitness of a chromosome.
        Lower cost = better fitness.
        
        Args:
            chromosome: Permutation of customer indices
            
        Returns:
            Total cost (fitness value)
        """
        routes = self._convert_to_routes(chromosome)
        return self._calculate_total_cost(routes)
    
    def _create_random_chromosome(self) -> List[int]:
        """Create a random chromosome (random permutation of customers)."""
        customers = list(range(self.num_locations))
        customers.remove(self.depot_idx)
        random.shuffle(customers)
        return customers
    
    def _create_greedy_chromosome(self) -> List[int]:
        """Create a chromosome using nearest neighbor heuristic."""
        unvisited = set(range(self.num_locations))
        unvisited.remove(self.depot_idx)
        
        chromosome = []
        current = self.depot_idx
        
        while unvisited:
            # Find nearest unvisited customer
            nearest = min(unvisited, key=lambda c: self.distance_matrix[current, c])
            chromosome.append(nearest)
            current = nearest
            unvisited.remove(nearest)
        
        return chromosome
    
    def initialize_population(self, pop_size: int, greedy_ratio: float = 0.5) -> List[List[int]]:
        """
        Initialize population with mix of random and greedy solutions.
        
        Args:
            pop_size: Population size
            greedy_ratio: Ratio of greedy solutions in initial population
            
        Returns:
            Initial population
        """
        population = []
        num_greedy = int(pop_size * greedy_ratio)
        
        # Add greedy solutions with some randomization
        for _ in range(num_greedy):
            chromosome = self._create_greedy_chromosome()
            # Add some randomness by swapping a few customers
            num_swaps = random.randint(1, 5)
            for _ in range(num_swaps):
                i, j = random.sample(range(len(chromosome)), 2)
                chromosome[i], chromosome[j] = chromosome[j], chromosome[i]
            population.append(chromosome)
        
        # Fill rest with random solutions
        for _ in range(pop_size - num_greedy):
            population.append(self._create_random_chromosome())
        
        return population
    
    def tournament_selection(self, population: List[List[int]], fitnesses: List[float], 
                           tournament_size: int = 3) -> List[int]:
        """
        Select a parent using tournament selection.
        
        Args:
            population: Current population
            fitnesses: Fitness values for each individual
            tournament_size: Number of individuals in tournament
            
        Returns:
            Selected parent chromosome
        """
        tournament_indices = random.sample(range(len(population)), tournament_size)
        tournament_fitnesses = [fitnesses[i] for i in tournament_indices]
        winner_idx = tournament_indices[tournament_fitnesses.index(min(tournament_fitnesses))]
        return population[winner_idx][:]
    
    def order_crossover(self, parent1: List[int], parent2: List[int]) -> Tuple[List[int], List[int]]:
        """
        Perform Order Crossover (OX) to create two offspring.
        
        Args:
            parent1: First parent chromosome
            parent2: Second parent chromosome
            
        Returns:
            Tuple of two offspring chromosomes
        """
        size = len(parent1)
        
        # Select two random crossover points
        cx_point1, cx_point2 = sorted(random.sample(range(size), 2))
        
        # Create offspring
        def create_offspring(p1, p2):
            offspring = [None] * size
            # Copy segment from parent1
            offspring[cx_point1:cx_point2] = p1[cx_point1:cx_point2]
            
            # Fill remaining positions from parent2
            p2_idx = cx_point2
            offspring_idx = cx_point2
            
            while None in offspring:
                if p2[p2_idx % size] not in offspring:
                    offspring[offspring_idx % size] = p2[p2_idx % size]
                    offspring_idx += 1
                p2_idx += 1
            
            return offspring
        
        offspring1 = create_offspring(parent1, parent2)
        offspring2 = create_offspring(parent2, parent1)
        
        return offspring1, offspring2
    
    def partially_mapped_crossover(self, parent1: List[int], parent2: List[int]) -> Tuple[List[int], List[int]]:
        """
        Perform Partially Mapped Crossover (PMX) to create two offspring.
        
        Args:
            parent1: First parent chromosome
            parent2: Second parent chromosome
            
        Returns:
            Tuple of two offspring chromosomes
        """
        size = len(parent1)
        
        # Select two random crossover points
        cx_point1, cx_point2 = sorted(random.sample(range(size), 2))
        
        def create_offspring(p1, p2):
            offspring = [None] * size
            # Copy segment from parent1
            offspring[cx_point1:cx_point2] = p1[cx_point1:cx_point2]
            
            # Create mapping
            for i in range(cx_point1, cx_point2):
                if p2[i] not in offspring:
                    # Find position for p2[i]
                    pos = i
                    while cx_point1 <= pos < cx_point2:
                        pos = p2.index(p1[pos])
                    offspring[pos] = p2[i]
            
            # Fill remaining positions
            for i in range(size):
                if offspring[i] is None:
                    offspring[i] = p2[i]
            
            return offspring
        
        offspring1 = create_offspring(parent1, parent2)
        offspring2 = create_offspring(parent2, parent1)
        
        return offspring1, offspring2
    
    def swap_mutation(self, chromosome: List[int]) -> List[int]:
        """
        Perform swap mutation: swap two random positions.
        
        Args:
            chromosome: Chromosome to mutate
            
        Returns:
            Mutated chromosome
        """
        mutated = chromosome[:]
        i, j = random.sample(range(len(mutated)), 2)
        mutated[i], mutated[j] = mutated[j], mutated[i]
        return mutated
    
    def inversion_mutation(self, chromosome: List[int]) -> List[int]:
        """
        Perform inversion mutation: reverse a random segment.
        
        Args:
            chromosome: Chromosome to mutate
            
        Returns:
            Mutated chromosome
        """
        mutated = chromosome[:]
        i, j = sorted(random.sample(range(len(mutated)), 2))
        mutated[i:j+1] = reversed(mutated[i:j+1])
        return mutated
    
    def scramble_mutation(self, chromosome: List[int]) -> List[int]:
        """
        Perform scramble mutation: shuffle a random segment.
        
        Args:
            chromosome: Chromosome to mutate
            
        Returns:
            Mutated chromosome
        """
        mutated = chromosome[:]
        i, j = sorted(random.sample(range(len(mutated)), 2))
        segment = mutated[i:j+1]
        random.shuffle(segment)
        mutated[i:j+1] = segment
        return mutated
    
    def apply_mutation(self, chromosome: List[int], mutation_rate: float) -> List[int]:
        """
        Apply mutation with given probability.
        
        Args:
            chromosome: Chromosome to mutate
            mutation_rate: Probability of mutation
            
        Returns:
            Possibly mutated chromosome
        """
        if random.random() < mutation_rate:
            # Randomly choose mutation type
            mutation_type = random.choice(['swap', 'inversion', 'scramble'])
            
            if mutation_type == 'swap':
                return self.swap_mutation(chromosome)
            elif mutation_type == 'inversion':
                return self.inversion_mutation(chromosome)
            else:
                return self.scramble_mutation(chromosome)
        
        return chromosome[:]
    
    def local_search_2opt(self, chromosome: List[int], max_improvements: int = 10) -> List[int]:
        """
        Apply 2-opt local search to a chromosome.
        
        Args:
            chromosome: Chromosome to improve
            max_improvements: Maximum number of improvements
            
        Returns:
            Improved chromosome
        """
        improved = chromosome[:]
        improvements = 0
        
        for _ in range(max_improvements):
            best_cost = self._evaluate_fitness(improved)
            found_improvement = False
            
            # Sample positions for large instances
            size = len(improved)
            step = max(1, size // 50)
            
            for i in range(0, size - 1, step):
                for j in range(i + 2, size, step):
                    # Reverse segment
                    new_chromosome = improved[:i] + improved[i:j][::-1] + improved[j:]
                    new_cost = self._evaluate_fitness(new_chromosome)
                    
                    if new_cost < best_cost:
                        improved = new_chromosome
                        best_cost = new_cost
                        improvements += 1
                        found_improvement = True
                        break
                
                if found_improvement:
                    break
            
            if not found_improvement:
                break
        
        return improved
    
    def solve(self, pop_size: int = 100, num_generations: int = 100, 
              crossover_rate: float = 0.8, mutation_rate: float = 0.2,
              elitism_count: int = 2, local_search_freq: int = 10,
              seed: Optional[int] = None) -> Tuple[List[List[int]], float]:
        """
        Solve VRP using Genetic Algorithm.
        
        Args:
            pop_size: Population size
            num_generations: Number of generations
            crossover_rate: Probability of crossover
            mutation_rate: Probability of mutation
            elitism_count: Number of best individuals to preserve
            local_search_freq: Apply local search every N generations
            seed: Random seed for reproducibility
            
        Returns:
            Tuple of (best routes, best cost)
        """
        if seed is not None:
            random.seed(seed)
            np.random.seed(seed)
        
        print(f"Starting Genetic Algorithm")
        print(f"Population size: {pop_size}")
        print(f"Generations: {num_generations}")
        print(f"Crossover rate: {crossover_rate}")
        print(f"Mutation rate: {mutation_rate}")
        print(f"Elitism: {elitism_count}")
        print(f"Number of customers: {self.num_locations - 1}")
        print(f"Vehicle capacity: {self.capacity}")
        print("-" * 60)
        
        # Initialize population
        population = self.initialize_population(pop_size)
        
        for generation in range(num_generations):
            # Evaluate fitness
            fitnesses = [self._evaluate_fitness(chrom) for chrom in population]
            
            # Track statistics
            best_fitness = min(fitnesses)
            avg_fitness = sum(fitnesses) / len(fitnesses)
            self.generation_costs.append(best_fitness)
            self.avg_costs.append(avg_fitness)
            
            # Update best solution
            if best_fitness < self.best_cost:
                best_idx = fitnesses.index(best_fitness)
                self.best_cost = best_fitness
                self.best_solution = self._convert_to_routes(population[best_idx])
                print(f"Generation {generation + 1}: New best cost = {best_fitness:.2f}")
            
            if (generation + 1) % 10 == 0:
                print(f"Generation {generation + 1}/{num_generations}: Best = {best_fitness:.2f}, Avg = {avg_fitness:.2f}")
            
            # Create new population
            new_population = []
            
            # Elitism: preserve best individuals
            sorted_indices = sorted(range(len(fitnesses)), key=lambda i: fitnesses[i])
            for i in range(elitism_count):
                new_population.append(population[sorted_indices[i]][:])
            
            # Generate offspring
            while len(new_population) < pop_size:
                # Selection
                parent1 = self.tournament_selection(population, fitnesses)
                parent2 = self.tournament_selection(population, fitnesses)
                
                # Crossover
                if random.random() < crossover_rate:
                    offspring1, offspring2 = self.order_crossover(parent1, parent2)
                else:
                    offspring1, offspring2 = parent1[:], parent2[:]
                
                # Mutation
                offspring1 = self.apply_mutation(offspring1, mutation_rate)
                offspring2 = self.apply_mutation(offspring2, mutation_rate)
                
                # Add to new population
                new_population.extend([offspring1, offspring2])
            
            # Trim to population size
            population = new_population[:pop_size]
            
            # Apply local search to best individuals periodically
            if (generation + 1) % local_search_freq == 0:
                fitnesses = [self._evaluate_fitness(chrom) for chrom in population]
                best_idx = fitnesses.index(min(fitnesses))
                population[best_idx] = self.local_search_2opt(population[best_idx])
        
        print("-" * 60)
        print(f"Genetic Algorithm completed!")
        print(f"Best cost found: {self.best_cost:.2f}")
        print(f"Number of routes: {len(self.best_solution)}")
        
        return self.best_solution, self.best_cost
    
    def print_solution(self):
        """Print the best solution in a readable format."""
        if self.best_solution is None:
            print("No solution available. Run solve() first.")
            return
        
        print("\n" + "=" * 60)
        print("GENETIC ALGORITHM VRP SOLUTION")
        print("=" * 60)
        print(f"Total cost: {self.best_cost:.2f}")
        print(f"Number of routes: {len(self.best_solution)}")
        print("\nRoutes:")
        
        for idx, route in enumerate(self.best_solution, 1):
            route_demand = sum(self.demands[c] for c in route)
            route_cost = self._calculate_route_cost(route)
            print(f"  Route {idx}: {[self.depot_idx] + route + [self.depot_idx]}")
            print(f"    Demand: {route_demand}/{self.capacity}, Cost: {route_cost:.2f}")
        
        print("=" * 60 + "\n")
    
    def plot_solution(self, output_path: Optional[str] = None):
        """
        Visualize the best solution.
        
        Args:
            output_path: Path to save the plot (optional)
        """
        if self.best_solution is None:
            print("No solution available. Run solve() first.")
            return
        
        plt.figure(figsize=(12, 10))
        
        # Plot depot
        depot_coord = self.coords[self.depot_idx]
        plt.scatter(depot_coord[0], depot_coord[1], c='red', s=200, marker='s', 
                   label='Depot', zorder=3)
        
        # Plot routes with different colors
        colors = plt.cm.tab20(np.linspace(0, 1, len(self.best_solution)))
        
        for idx, (route, color) in enumerate(zip(self.best_solution, colors), 1):
            # Plot route
            route_coords = [depot_coord] + [self.coords[c] for c in route] + [depot_coord]
            route_coords = np.array(route_coords)
            
            plt.plot(route_coords[:, 0], route_coords[:, 1], 'o-', 
                    color=color, linewidth=2, markersize=6, label=f'Route {idx}')
        
        plt.xlabel('X Coordinate', fontsize=12)
        plt.ylabel('Y Coordinate', fontsize=12)
        plt.title(f'Genetic Algorithm VRP Solution (Cost: {self.best_cost:.2f})', 
                 fontsize=14, fontweight='bold')
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        
        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"Plot saved to {output_path}")
        
        plt.show()
    
    def plot_convergence(self, output_path: Optional[str] = None):
        """
        Plot the convergence of the Genetic Algorithm.
        
        Args:
            output_path: Path to save the plot (optional)
        """
        if not self.generation_costs:
            print("No convergence data available. Run solve() first.")
            return
        
        plt.figure(figsize=(10, 6))
        
        generations = range(1, len(self.generation_costs) + 1)
        plt.plot(generations, self.generation_costs, 'b-', linewidth=2, label='Best Cost')
        plt.plot(generations, self.avg_costs, 'g--', alpha=0.6, label='Average Cost')
        
        plt.xlabel('Generation', fontsize=12)
        plt.ylabel('Solution Cost', fontsize=12)
        plt.title('Genetic Algorithm Convergence', fontsize=14, fontweight='bold')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        
        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"Convergence plot saved to {output_path}")
        
        plt.show()
    
    def plot_diversity(self, output_path: Optional[str] = None):
        """
        Plot population diversity over generations.
        
        Args:
            output_path: Path to save the plot (optional)
        """
        if not self.avg_costs or not self.generation_costs:
            print("No diversity data available. Run solve() first.")
            return
        
        plt.figure(figsize=(10, 6))
        
        # Calculate diversity as difference between average and best
        diversity = [avg - best for avg, best in zip(self.avg_costs, self.generation_costs)]
        generations = range(1, len(diversity) + 1)
        
        plt.plot(generations, diversity, 'purple', linewidth=2)
        plt.xlabel('Generation', fontsize=12)
        plt.ylabel('Population Diversity (Avg - Best)', fontsize=12)
        plt.title('Population Diversity Over Time', fontsize=14, fontweight='bold')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        
        if output_path:
            plt.savefig(output_path, dpi=300, bbox_inches='tight')
            print(f"Diversity plot saved to {output_path}")
        
        plt.show()


def main():
    """
    Main function to solve VRP using Genetic Algorithm.
    Runs on the full 1000+ node instance.
    """
    import time
    import vrplib
    from pathlib import Path
    
    instance_path = "../data/instance.vrp"
    
    print("\n" + "=" * 80)
    print("GENETIC ALGORITHM METAHEURISTIC - VRP with 1 Truck (Multiple Trips)")
    print("=" * 80)
    print(f"Instance: {instance_path}")
    print(f"Configuration: Population=100, Generations=100, Capacity=100")
    print("=" * 80 + "\n")
    
    # Check if file exists
    if not Path(instance_path).exists():
        print(f"ERROR: Instance file not found: {instance_path}")
        print("Using small example instead...")
        
        # Run GA with example data
        coords = np.array([
            [50, 50],  # Depot
            [23, 67], [78, 34], [12, 89], [45, 23], [67, 56],
            [34, 78], [89, 12], [56, 45], [90, 67], [34, 23]
        ])
        demands = np.array([0, 5, 8, 3, 7, 4, 6, 9, 2, 5, 7])
        capacity = 20
    else:
        # Parse full instance
        print("Loading instance...")
        data = vrplib.read_instance(instance_path)
        coords = np.array(data['node_coord'])
        dimension = data['dimension']
        capacity = data.get('capacity', 100)
        
        # Get demands if available, otherwise default to 1 for all customers
        if 'demand' in data:
            demands = np.array(data['demand'])
        else:
            demands = np.ones(dimension, dtype=int)
            demands[0] = 0  # Depot has 0 demand
        
        print(f"Loaded: {dimension} nodes, capacity {capacity}")
        print(f"Total demand: {demands.sum()}")
        print()
    
    start_time = time.time()
    
    # Create and run GA solver
    solver = GeneticAlgorithmSolver(coords, demands, capacity, depot_idx=0)
    solution, cost = solver.solve(
        pop_size=100,
        num_generations=100,
        crossover_rate=0.8,
        mutation_rate=0.2,
        elitism_count=2,
        local_search_freq=10,
        seed=42
    )
    
    # Display results
    solver.print_solution()
    
    elapsed_time = time.time() - start_time
    print(f"Total execution time: {elapsed_time:.2f} seconds")
    
    # Try to create visualizations
    try:
        print("\nGenerating visualizations...")
        solver.plot_solution(output_path="../data/ga_solution.png")
        solver.plot_convergence(output_path="../data/ga_convergence.png")
        solver.plot_diversity(output_path="../data/ga_diversity.png")
    except Exception as e:
        print(f"Could not create plots: {e}")
    
    print("=" * 80 + "\n")
    
    return solver


if __name__ == "__main__":
    main()



GENETIC ALGORITHM METAHEURISTIC - VRP with 1 Truck (Multiple Trips)
Instance: ../data/instance.vrp
Configuration: Population=100, Generations=100, Capacity=100

Loading instance...
Loaded: 1001 nodes, capacity 1000
Total demand: 1000

Starting Genetic Algorithm
Population size: 100
Generations: 100
Crossover rate: 0.8
Mutation rate: 0.2
Elitism: 2
Number of customers: 1000
Vehicle capacity: 1000
------------------------------------------------------------
Generation 1: New best cost = 1780.85
Generation 10/100: Best = 1780.85, Avg = 2980.48
Generation 20/100: Best = 1780.85, Avg = 2058.14
Generation 30/100: Best = 1780.85, Avg = 1844.59
Generation 40/100: Best = 1780.85, Avg = 1953.38
Generation 50/100: Best = 1780.85, Avg = 3277.49
Generation 60/100: Best = 1780.85, Avg = 2982.80
Generation 70/100: Best = 1780.85, Avg = 2423.62
Generation 80/100: Best = 1780.85, Avg = 2009.66
Generation 90/100: Best = 1780.85, Avg = 2448.69
Generation 100/100: Best = 1780.85, Avg = 2361.27
----------

Convergence plot saved to ../data/ga_convergence.png


Diversity plot saved to ../data/ga_diversity.png


### 7.3 VRP Reference Solver

The reference solver provides an interface to state-of-the-art exact and metaheuristic solving algorithms for VRP instances.

#### Core Functionalities

**Instance Loading**: Reads problem instances in standard VRPLIB format, extracting customer locations, distance matrices, and problem constraints.

**Configurable Solving**: Allows specification of maximum runtime limits, enabling fair comparison with GA under identical time budgets.

**Solution Export**: Saves computed solutions in standard solution format (`.sol`) for verification and reproducibility.

**Performance Statistics**: Returns detailed solving statistics including objective value, computational time, and solution quality metrics.

In [2]:
"""
VRP Solver using PyVRP library.
This module provides functionality to solve Vehicle Routing Problems using the PyVRP library.
"""

import numpy as np
from pathlib import Path
from pyvrp import Model, read
from pyvrp.stop import MaxRuntime


class VRPSolver:
    """Solver for Vehicle Routing Problems using PyVRP."""
    
    def __init__(self, instance_path: str):
        """
        Initialize the VRP solver with an instance file.
        
        Args:
            instance_path: Path to the VRP instance file
        """
        self.instance_path = Path(instance_path)
        self.instance_data = None
        self.model = None
        self.result = None
        
    def load_instance(self):
        """Load the VRP instance from file using PyVRP's read function."""
        print(f"Loading instance from {self.instance_path}")
        # Use PyVRP's native read function which properly handles VRPLIB format
        self.instance_data = read(str(self.instance_path), round_func='round')
        return self.instance_data
    
    def build_model(self):
        """Build the PyVRP model from the loaded instance."""
        if self.instance_data is None:
            self.load_instance()
        
        # Use PyVRP's from_data() method with the ProblemData object
        self.model = Model.from_data(self.instance_data)
        
        print(f"Model built from PyVRP data")
        print(f"  Locations: {self.instance_data.num_locations}")
        print(f"  Vehicles: {self.instance_data.num_vehicle_types}")
        
        return self.model
    
    def solve(self, time_limit: int = 180, seed: int = 42):
        """
        Solve the VRP instance.
        
        Args:
            time_limit: Maximum runtime in seconds (default: 180)
            seed: Random seed for reproducibility
            
        Returns:
            Result object from PyVRP
        """
        if self.model is None:
            self.build_model()
        
        print(f"Solving VRP with time limit: {time_limit}s")
        
        # Solve the problem
        self.result = self.model.solve(
            stop=MaxRuntime(time_limit),
            seed=seed
        )
        
        # Check if solution is feasible
        print(f"\nSolution statistics:")
        print(f"  Cost: {self.result.cost()}")
        print(f"  Feasible: {self.result.is_feasible()}")
        
        # Debug: print result attributes
        print(f"\nResult object attributes: {[attr for attr in dir(self.result) if not attr.startswith('_')]}")
        
        if self.result.is_feasible():
            # Try to get basic stats
            try:
                best = self.result.best
                print(f"  Number of routes in solution: {best.num_routes()}")
                print(f"  Routes: {list(best.routes())}")
            except Exception as e:
                print(f"  Could not get route info: {e}")
        else:
            print(f"WARNING: No feasible solution found within time limit")
        
        return self.result
    
    def get_routes(self):
        """
        Get the routes from the solution.
        
        Returns:
            List of routes, where each route is a list of client indices
        """
        if self.result is None:
            raise ValueError("No solution available. Run solve() first.")
        
        # Get routes from the result object using best() method
        routes = []
        try:
            best_solution = self.result.best
            for route in best_solution.routes():
                route_clients = [visit for visit in route]
                routes.append(route_clients)
        except Exception as e:
            print(f"Error extracting routes: {e}")
            # Try alternative approach
            try:
                import traceback
                traceback.print_exc()
            except:
                pass
        return routes
    
    def print_solution(self):
        """Print the solution in a readable format."""
        if self.result is None:
            print("No solution available.")
            return
        
        print("\n" + "=" * 60)
        print("VRP SOLUTION")
        print("=" * 60)
        print(f"Total cost: {self.result.cost()}")
        print(f"Is feasible: {self.result.is_feasible()}")
        
        if self.result.is_feasible():
            try:
                routes = self.get_routes()
                print(f"Number of routes: {len(routes)}")
                print("\nRoutes:")
                for idx, route in enumerate(routes, 1):
                    print(f"  Route {idx}: {route[:10]}..." if len(route) > 10 else f"  Route {idx}: {route}")
            except Exception as e:
                print(f"Could not display routes: {e}")
        else:
            print("No feasible solution found.")
        
        print("=" * 60 + "\n")
    
    def save_solution(self, output_path: str):
        """
        Save the solution to a file in VRPLIB .sol format.
        
        Args:
            output_path: Path to save the solution
        """
        if self.result is None:
            raise ValueError("No solution available. Run solve() first.")
        
        with open(output_path, 'w') as f:
            # VRPLIB .sol format
            if self.result.is_feasible():
                try:
                    routes = self.get_routes()
                    # Write cost (rounded to integer for VRPLIB format)
                    f.write(f"{int(self.result.cost())}\n")
                    
                    # Write each route
                    for route in routes:
                        # VRPLIB format: Route #: node1 node2 node3 ...
                        # Client indices in PyVRP start from 1 (depot is 0)
                        route_str = ' '.join(map(str, route))
                        f.write(f"Route #{len([r for r in routes[:routes.index(route)+1]])}: {route_str}\n")
                except Exception as e:
                    f.write(f"Error: Could not extract routes: {e}\n")
            else:
                f.write("No feasible solution found.\n")
        
        print(f"Solution saved to {output_path} (VRPLIB .sol format)")


def main():
    """
    Main function to solve VRP using PyVRP.
    
    Note: With 1000 customers each having demand=1 and capacity=1000,
    all customers can fit in a single route (essentially a TSP).
    """
    import time
    
    # Path to instance file
    instance_file = "../data/instance.vrp"
    
    print("\n" + "=" * 80)
    print("PYVRP SOLVER - VRP with 1 Truck (Multiple Trips)")
    print("=" * 80)
    print(f"Instance: {instance_file}")
    print(f"Time limit: 180 seconds")
    print(f"Configuration: 1 truck, capacity 1000 (all customers in 1 route)")
    print("=" * 80 + "\n")
    
    start_time = time.time()
    
    # Create solver
    solver = VRPSolver(instance_file)
    
    # Load and solve
    solver.load_instance()
    solver.build_model()
    result = solver.solve(time_limit=180, seed=42)
    
    # Display results
    solver.print_solution()
    
    # Save solution in VRPLIB .sol format
    solver.save_solution("../data/pyvrp_solution.sol")
    
    elapsed_time = time.time() - start_time
    print(f"\nTotal execution time: {elapsed_time:.2f} seconds")
    print("=" * 80 + "\n")
    
    return solver


if __name__ == "__main__":
    main()



PYVRP SOLVER - VRP with 1 Truck (Multiple Trips)
Instance: ../data/instance.vrp
Time limit: 180 seconds
Configuration: 1 truck, capacity 1000 (all customers in 1 route)

Loading instance from ..\data\instance.vrp
Model built from PyVRP data
  Locations: 1001
  Vehicles: 1
Solving VRP with time limit: 180s
PyVRP v0.12.1

Solving an instance with:
    1 depot
    1000 clients
    1000 vehicles (1 vehicle type)

                  |       Feasible        |      Infeasible
    Iters    Time |   #      Avg     Best |   #      Avg     Best
H     529      5s |  62     2546     1461 |   0        -        -
     1237     10s |  32     3494     1461 |   0        -        -
     1954     15s |  52     2735     1461 |   0        -        -
     2675     20s |  35     3328     1461 |   0        -        -
     3369     25s |  32     3495     1461 |   0        -        -
     4093     30s |  59     2590     1461 |   0        -        -
     4845     35s |  32     3493     1461 |   0        -        

### 7.4 Solution Verification and Comparison Framework

The comparison framework provides comprehensive evaluation of GA performance against reference solutions through multiple verification and analysis steps.

#### Verification Procedures

**Solution Feasibility Check**: Uses standard VRP verification libraries to confirm that solutions satisfy all problem constraints (all customers visited exactly once, vehicle capacity limits respected, etc.).

**Cost Computation**: Independently computes the objective value of each solution to ensure accuracy and consistency.

#### Comparative Analysis

**Side-by-side Comparison**: Evaluates both GA and reference solutions on identical problem instances, recording their respective costs and runtimes.

**Gap Analysis**: Calculates the percentage difference between GA and reference solutions, quantifying solution quality relative to known baselines.

**Visual Route Comparison**: Generates graphical representations of routes computed by both algorithms, enabling visual inspection of solution characteristics.

#### Performance Metrics

The framework computes multiple performance indicators:
- **Solution cost**: Total distance or delivery cost
- **Computational time**: Runtime in seconds
- **Optimality gap**: Percentage difference from reference solution
- **Convergence rate**: Number of generations to reach the best solution

In [3]:
"""
Compare solutions from Genetic Algorithm and PyVRP solver.
This module verifies and compares the solutions from both algorithms.
"""

import vrplib
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt


def verify_solution(instance_file: str, solution_file: str):
    """
    Verify a solution using vrplib.
    
    Args:
        instance_file: Path to the VRP instance file
        solution_file: Path to the solution file (.sol format)
        
    Returns:
        dict: Verification results with cost, feasibility, etc.
    """
    print(f"\nVerifying solution: {solution_file}")
    print("-" * 60)
    
    # Read instance and solution
    instance = vrplib.read_instance(instance_file)
    solution = vrplib.read_solution(solution_file)
    
    print(f"Solution loaded from file:")
    print(f"  Routes: {len(solution['routes'])}")
    
    # Get instance parameters
    coords = instance['node_coord']
    capacity = instance['capacity']
    demands = instance.get('demand', [0] + [1] * (instance['dimension'] - 1))
    
    # Manually compute cost and check feasibility
    total_cost = 0
    is_feasible = True
    
    for idx, route in enumerate(solution['routes'], 1):
        # Calculate route distance
        route_cost = 0
        # Start from depot (node 0)
        prev_node = 0
        for node in route:
            dist = np.linalg.norm(coords[prev_node] - coords[node])
            route_cost += dist
            prev_node = node
        # Return to depot
        route_cost += np.linalg.norm(coords[prev_node] - coords[0])
        
        # Calculate route demand
        route_demand = sum(demands[node] for node in route)
        
        # Check feasibility
        if route_demand > capacity:
            is_feasible = False
            print(f"  Route {idx}: {len(route)} customers, demand={route_demand}/{capacity} - INFEASIBLE!")
        else:
            print(f"  Route {idx}: {len(route)} customers, demand={route_demand}/{capacity}, cost={route_cost:.2f}")
        
        total_cost += route_cost
    
    print(f"\nComputed total cost: {total_cost:.2f}")
    print(f"Feasible: {is_feasible}")
    
    return {
        'cost': total_cost,
        'feasible': is_feasible,
        'num_routes': len(solution['routes']),
        'solution': solution
    }


def save_ga_solution_as_sol(ga_solver, output_path: str):
    """
    Save GA solution in VRPLIB .sol format.
    
    Args:
        ga_solver: GeneticAlgorithmSolver instance with solution
        output_path: Path to save the .sol file
    """
    with open(output_path, 'w') as f:
        # Write cost
        f.write(f"{int(ga_solver.best_cost)}\n")
        
        # Write routes
        routes = ga_solver.best_solution  # Use best_solution directly
        for idx, route in enumerate(routes, 1):
            # VRPLIB format: Route #X: node1 node2 node3 ...
            route_str = ' '.join(map(str, route))
            f.write(f"Route #{idx}: {route_str}\n")
    
    print(f"GA solution saved to {output_path}")


def visualize_comparison(ga_results: dict, pyvrp_results: dict, save_path: str = None):
    """
    Create visualization comparing GA and PyVRP results.
    
    Args:
        ga_results: GA verification results
        pyvrp_results: PyVRP verification results
        save_path: Optional path to save the figure
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Cost comparison
    algorithms = ['Genetic Algorithm', 'PyVRP']
    costs = [ga_results['cost'], pyvrp_results['cost']]
    colors = ['#2ecc71', '#3498db']
    
    axes[0].bar(algorithms, costs, color=colors, alpha=0.7, edgecolor='black')
    axes[0].set_ylabel('Total Cost', fontsize=12)
    axes[0].set_title('Solution Cost Comparison', fontsize=14, fontweight='bold')
    axes[0].grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for i, (alg, cost) in enumerate(zip(algorithms, costs)):
        axes[0].text(i, cost, f'{int(cost)}', ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    # Routes comparison
    num_routes = [ga_results['num_routes'], pyvrp_results['num_routes']]
    
    axes[1].bar(algorithms, num_routes, color=colors, alpha=0.7, edgecolor='black')
    axes[1].set_ylabel('Number of Routes', fontsize=12)
    axes[1].set_title('Number of Routes Comparison', fontsize=14, fontweight='bold')
    axes[1].grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for i, (alg, routes) in enumerate(zip(algorithms, num_routes)):
        axes[1].text(i, routes, f'{routes}', ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Comparison plot saved to {save_path}")
    
    plt.show()


def main():
    """Main comparison function."""
    print("\n" + "=" * 80)
    print("ALGORITHM COMPARISON - Genetic Algorithm vs PyVRP")
    print("=" * 80)
    
    # Configure instance file here - change this to test different instances
    instance_file = "../data/instance.vrp" 
    ga_sol_file = "../data/ga_solution.sol"
    pyvrp_sol_file = "../data/pyvrp_solution.sol"
    
    # Run Genetic Algorithm
    print("\n" + "=" * 80)
    print("RUNNING GENETIC ALGORITHM")
    print("=" * 80)
    
    # Load instance data
    instance = vrplib.read_instance(instance_file)
    coords = instance['node_coord']
    capacity = instance['capacity']
    demands = instance.get('demand', [0] + [1] * (instance['dimension'] - 1))
    
    ga_solver = GeneticAlgorithmSolver(coords, demands, capacity)
    ga_solver.solve(
        pop_size=320, #320 for 1000 / 1400 for 200 // change as needed -> here for 180s computation time
        num_generations=200, #200 for 1000 / 1000 for 200 // change as needed -> here for 180s computation time
        crossover_rate=0.6,
        mutation_rate=0.05,
        elitism_count=10,
        local_search_freq=4,
        seed=42
    )
    
    # Save GA solution in .sol format
    save_ga_solution_as_sol(ga_solver, ga_sol_file)
    
    # Verify GA solution
    ga_results = verify_solution(instance_file, ga_sol_file)
    
    print("\n" + "=" * 80)
    print("GA SOLUTION SUMMARY")
    print("=" * 80)
    print(f"Best cost: {ga_solver.best_cost:.2f}")
    print(f"Number of routes: {len(ga_solver.best_solution)}")
    print(f"Verified cost: {ga_results['cost']}")
    print(f"Feasible: {ga_results['feasible']}")
    
    # Run PyVRP solver
    print("\n" + "=" * 80)
    print("RUNNING PYVRP SOLVER")
    print("=" * 80)
    
    pyvrp_solver = VRPSolver(instance_file)
    pyvrp_solver.load_instance()
    pyvrp_solver.build_model()
    pyvrp_solver.solve(time_limit=120, seed=42)
    pyvrp_solver.save_solution(pyvrp_sol_file)
    
    # Verify PyVRP solution
    print("\n" + "=" * 80)
    print("VERIFYING PYVRP SOLUTION")
    print("=" * 80)
    
    pyvrp_results = verify_solution(instance_file, pyvrp_sol_file)
    
    # Comparison
    print("\n" + "=" * 80)
    print("COMPARISON RESULTS")
    print("=" * 80)
    
    print(f"\nGenetic Algorithm:")
    print(f"  Cost: {ga_results['cost']:.2f}")
    print(f"  Routes: {ga_results['num_routes']}")
    print(f"  Feasible: {ga_results['feasible']}")
    
    print(f"\nPyVRP:")
    print(f"  Cost: {pyvrp_results['cost']:.2f}")
    print(f"  Routes: {pyvrp_results['num_routes']}")
    print(f"  Feasible: {pyvrp_results['feasible']}")
    
    # Calculate difference
    cost_diff = ga_results['cost'] - pyvrp_results['cost']
    cost_diff_pct = (cost_diff / pyvrp_results['cost']) * 100
    
    print(f"\nDifference:")
    print(f"  Cost difference: {cost_diff:+.2f} ({cost_diff_pct:+.2f}%)")
    
    if ga_results['cost'] < pyvrp_results['cost']:
        print(f"  ✓ GA found a better solution!")
    elif ga_results['cost'] > pyvrp_results['cost']:
        print(f"  ✓ PyVRP found a better solution!")
    else:
        print(f"  ✓ Both algorithms found the same cost!")
    
    # Visualize comparison
    print("\n" + "=" * 80)
    print("GENERATING COMPARISON VISUALIZATION")
    print("=" * 80)
    
    visualize_comparison(ga_results, pyvrp_results, save_path="../data/algorithm_comparison.png")
    
    print("\n" + "=" * 80)
    print("COMPARISON COMPLETE")
    print("=" * 80 + "\n")


if __name__ == "__main__":
    main()



ALGORITHM COMPARISON - Genetic Algorithm vs PyVRP

RUNNING GENETIC ALGORITHM
Starting Genetic Algorithm
Population size: 320
Generations: 200
Crossover rate: 0.6
Mutation rate: 0.05
Elitism: 10
Number of customers: 1000
Vehicle capacity: 1000
------------------------------------------------------------
Generation 1: New best cost = 1780.85
Generation 4: New best cost = 1777.53
Generation 5: New best cost = 1774.57
Generation 10/200: Best = 1774.57, Avg = 1876.56
Generation 20/200: Best = 1774.57, Avg = 1855.47
Generation 30/200: Best = 1774.57, Avg = 1801.54
Generation 40/200: Best = 1774.57, Avg = 1862.56
Generation 50/200: Best = 1774.57, Avg = 1834.26
Generation 60/200: Best = 1774.57, Avg = 1999.99
Generation 70/200: Best = 1774.57, Avg = 2056.72
Generation 80/200: Best = 1774.57, Avg = 1848.65
Generation 90/200: Best = 1774.57, Avg = 1905.03
Generation 100/200: Best = 1774.57, Avg = 1940.45
Generation 110/200: Best = 1774.57, Avg = 1953.12
Generation 120/200: Best = 1774.57, Avg 


COMPARISON COMPLETE



## 8. Design of Experiments

This section presents the experimental methodology for evaluating GA performance, analyzing parameter sensitivity, and validating solution quality through rigorous statistical analysis.

### 8.1 Experimental Objectives

The experimental design addresses the following research questions:

1. **Performance Assessment**: How does GA solution quality compare to state-of-the-art exact and heuristic methods across different problem sizes and characteristics?

2. **Parameter Sensitivity Analysis**: Which GA parameters most significantly influence solution quality and convergence behavior? What are the optimal ranges for each parameter?

3. **Stochastic Variability Quantification**: How much variation exists in GA solutions across multiple independent runs? Which configurations produce most consistent results?

4. **Statistical Significance Testing**: Do observed differences between parameter configurations represent true performance differences or occur due to random variation?

5. **Optimal Configuration Identification**: Which combination of parameter settings consistently produces superior solutions within acceptable runtime constraints?

### 8.2 Experimental Factors and Ranges

The experimental design systematically varies the following parameters:

#### Genetic Algorithm Parameters

**Population Size**: Controls the number of candidate solutions maintained in each generation. Larger populations increase exploration but require more computation. Tested values: 50, 100, 200 individuals.

**Mutation Rate**: Specifies the probability of applying swap mutation to each offspring. Low values may cause premature convergence to suboptimal solutions; high values may prevent convergence to good solutions. Tested values: 0.01, 0.05, 0.1.

**Crossover Rate**: Determines the probability of applying Order Crossover to parent pairs. Controls balance between exploitation of good solutions and exploration of new regions. Tested values: 0.7, 0.8, 0.9.

**Tournament Size**: Sets the number of individuals competing in each tournament selection round. Larger tournaments increase selection pressure toward best solutions. Tested values: 2, 3, 5.

**Elite Size**: Specifies the number of best individuals preserved unchanged to the next generation. Ensures solution quality never degrades. Tested values: 1, 2, 5.

**Maximum Generations**: Establishes upper bound on the number of evolutionary cycles. Tested values: 500, 1000 generations.

**Stagnation Limit**: Defines the maximum number of generations without improvement before terminating evolution. Tested values: 50, 100, 200 generations.

#### Performance Metrics

The evaluation measures the following aspects of algorithm performance:

- **Solution Cost**: Total distance or delivery cost of computed routes
- **Computational Runtime**: Wall-clock time in seconds required to reach the best solution
- **Optimality Gap**: Percentage difference between GA solution and reference solution cost
- **Convergence Speed**: Number of generations required to discover the best solution

### 8.3 Statistical Analysis Module

The statistical analysis component executes multiple independent GA runs and aggregates results to characterize performance distributions.

#### Multiple Independent Runs

Each parameter configuration is executed 20 independent times with identical parameters but different random seeds. This captures the stochastic nature of the genetic algorithm and enables robust statistical characterization.

#### Performance Distribution Visualization

**Boxplot Construction**: Results from 20 runs are visualized as boxplots displaying:
- **Median**: Central line representing typical performance
- **Interquartile Range**: Box boundaries showing 25th and 75th percentiles (middle 50% of results)
- **Whiskers**: Extend to show range excluding extreme outliers
- **Outliers**: Individual points represent exceptionally good or poor runs

#### Configuration Comparison

Multiple configurations can be visualized side-by-side, enabling direct visual comparison of:
- Central tendency (median performance)
- Variability (spread and consistency)
- Best-case performance (upper whisker/maximum)
- Worst-case performance (lower whisker/minimum)
- Stability (consistency across runs)

This statistical approach enables identification of configurations that consistently produce high-quality solutions versus those with high variance.

In [4]:
import numpy as np
import matplotlib.pyplot as plt


class Boxplot20:
    
    def __init__(self):
        self.nmber_of_runs = 20
        self.results = []
        self.labels = []
        self.colors = ['blue', 'green', 'red', 'purple', 'orange']
        self.title = "GA VRP Solution Costs over 20 Runs"
        self.ylabel = "Solution Cost"
        self.xlabel = "GA Configurations"
        self.fig, self.ax = plt.subplots(figsize=(10, 6))

    def add_results(self, method_name: str, costs: list):
        self.results.append(costs)
        self.labels.append(method_name)

    def plot(self, save_path: str = None):
        """Plot boxplot and optionally save to file."""
        self.ax.boxplot(self.results, tick_labels=self.labels)
        self.ax.set_title(self.title)
        self.ax.set_ylabel(self.ylabel)
        self.ax.set_xlabel(self.xlabel)
        self.ax.grid(axis='y', alpha=0.3)
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"Boxplot saved to {save_path}")
        
        plt.show()
    
    def plot_bar_chart(self, save_path: str = None):
        """Plot bar chart showing individual run costs."""
        fig, ax = plt.subplots(figsize=(14, 6))
        
        # For each configuration, plot bars for all runs
        bar_width = 0.8 / len(self.results)
        x_positions = np.arange(self.nmber_of_runs)
        
        for idx, (label, costs) in enumerate(zip(self.labels, self.results)):
            offset = (idx - len(self.results)/2 + 0.5) * bar_width
            bars = ax.bar(x_positions + offset, costs, bar_width, 
                          label=label, alpha=0.7, 
                          color=self.colors[idx % len(self.colors)])
            
            # Add mean line
            mean_cost = np.mean(costs)
            ax.axhline(y=mean_cost, color=self.colors[idx % len(self.colors)], 
                      linestyle='--', linewidth=2, alpha=0.5,
                      label=f'{label} Mean: {mean_cost:.2f}')
        
        ax.set_xlabel('Run Number', fontsize=12)
        ax.set_ylabel('Solution Cost', fontsize=12)
        ax.set_title('GA Solution Costs - Individual Runs', fontsize=14, fontweight='bold')
        ax.set_xticks(x_positions)
        ax.set_xticklabels([f'{i+1}' for i in range(self.nmber_of_runs)])
        ax.legend()
        ax.grid(axis='y', alpha=0.3)
        
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"Bar chart saved to {save_path}")
        
        plt.show()
    
    def save_statistics(self, save_path: str = None):
        """Print and optionally save statistics for each configuration."""
        print("\n" + "=" * 80)
        print("STATISTICAL SUMMARY")
        print("=" * 80)
        
        stats_lines = []
        for label, costs in zip(self.labels, self.results):
            print(f"\n{label}:")
            print(f"  Mean:   {np.mean(costs):.2f}")
            print(f"  Median: {np.median(costs):.2f}")
            print(f"  Std:    {np.std(costs):.2f}")
            print(f"  Min:    {np.min(costs):.2f}")
            print(f"  Max:    {np.max(costs):.2f}")
            
            stats_lines.append(f"{label}:")
            stats_lines.append(f"  Mean:   {np.mean(costs):.2f}")
            stats_lines.append(f"  Median: {np.median(costs):.2f}")
            stats_lines.append(f"  Std:    {np.std(costs):.2f}")
            stats_lines.append(f"  Min:    {np.min(costs):.2f}")
            stats_lines.append(f"  Max:    {np.max(costs):.2f}")
            stats_lines.append("")
        
        if save_path:
            with open(save_path, 'w') as f:
                f.write('\n'.join(stats_lines))
            print(f"\nStatistics saved to {save_path}")


def main():
    """Run GA 20 times without seed and generate boxplot."""
    import vrplib
    
    print("\n" + "=" * 80)
    print("GA BOXPLOT ANALYSIS - 20 Runs Without Seed")
    print("=" * 80)
    
    # Configure instance file
    instance_file = "../data/200instance.vrp"
    
    print(f"Instance: {instance_file}")
    print(f"Number of runs: 20")
    print(f"Seed: None (random runs)")
    print("=" * 80 + "\n")
    
    # Load instance
    instance = vrplib.read_instance(instance_file)
    coords = instance['node_coord']
    capacity = instance['capacity']
    demands = instance.get('demand', [0] + [1] * (instance['dimension'] - 1))
    
    # Create boxplot analyzer
    boxplot = Boxplot20()
    
    # Run GA 20 times without seed
    print("Running GA 20 times (this may take a while)...\n")
    costs = []
    
    for run in range(1, 21):
        print(f"Run {run}/20...", end=" ", flush=True)
        
        solver = GeneticAlgorithmSolver(coords, demands, capacity)
        solution, cost = solver.solve(
            pop_size=100,
            num_generations=300,
            crossover_rate=0.6,
            mutation_rate=0.05,
            elitism_count=10,
            local_search_freq=4,
            seed=None  # No seed for random runs
        )
        
        costs.append(cost)
        print(f"Cost: {cost:.2f}")
    
    # Add results to boxplot
    boxplot.add_results("GA (20 runs)", costs)
    
    # Display statistics
    boxplot.save_statistics()
    
    # Generate and save boxplot
    print("\n" + "=" * 80)
    print("GENERATING BOXPLOT")
    print("=" * 80)
    
    boxplot.plot(save_path="../data/ga_boxplot_20runs.png")
    
    # Generate and save bar chart
    print("\n" + "=" * 80)
    print("GENERATING BAR CHART")
    print("=" * 80)
    
    boxplot.plot_bar_chart(save_path="../data/ga_barchart_20runs.png")
    
    print("\n" + "=" * 80)
    print("ANALYSIS COMPLETE")
    print("=" * 80 + "\n")


if __name__ == "__main__":
    main()
        


GA BOXPLOT ANALYSIS - 20 Runs Without Seed
Instance: ../data/200instance.vrp
Number of runs: 20
Seed: None (random runs)

Running GA 20 times (this may take a while)...

Run 1/20... Starting Genetic Algorithm
Population size: 100
Generations: 300
Crossover rate: 0.6
Mutation rate: 0.05
Elitism: 10
Number of customers: 200
Vehicle capacity: 200
------------------------------------------------------------
Generation 1: New best cost = 1208.60
Generation 2: New best cost = 1202.22
Generation 3: New best cost = 1180.04
Generation 5: New best cost = 1155.10
Generation 9: New best cost = 1149.28
Generation 10/300: Best = 1149.28, Avg = 1217.58
Generation 20/300: Best = 1149.28, Avg = 1188.27
Generation 30/300: Best = 1149.28, Avg = 1155.59
Generation 40/300: Best = 1149.28, Avg = 1158.42
Generation 50/300: Best = 1149.28, Avg = 1150.79
Generation 60/300: Best = 1149.28, Avg = 1162.20
Generation 70/300: Best = 1149.28, Avg = 1219.93
Generation 80/300: Best = 1149.28, Avg = 1151.08
Generation


GENERATING BAR CHART
Bar chart saved to ../data/ga_barchart_20runs.png



ANALYSIS COMPLETE



### 8.4 Expected Outcomes and Performance Analysis

#### Theoretical Performance Predictions

Based on established genetic algorithm theory and VRP literature, the following outcomes are expected:

**Population Size Effect**: Larger populations provide better exploration of solution space, typically resulting in higher-quality solutions but requiring proportionally more computation time. A threshold effect may occur where doubling population size yields diminishing quality improvements.

**Mutation Rate Trade-off**: Very low mutation rates (< 0.01) risk premature convergence to local optima. Very high rates (> 0.2) introduce excessive disruption, preventing convergence to good solutions. Intermediate values (0.05-0.1) typically balance exploration and exploitation effectively.

**Crossover Operator Effectiveness**: Order Crossover effectively preserves route structure from both parents while allowing diversity. High crossover rates (0.8-0.9) should generally outperform low rates by maximizing beneficial genetic recombination.

**Elitism Benefit**: Preserving elite solutions should accelerate convergence toward the best discovered solution and ensure monotonic improvement across generations.

**Stochastic Variability**: Multiple runs will reveal significant variance in solution costs, with distributions characterized by the boxplot statistics. More robust configurations should exhibit smaller interquartile ranges.

#### Expected Solution Quality

For small problem instances (< 50 customers), GA solutions are expected to fall within 5-15% of optimal solutions provided by exact methods. As problem size increases, the absolute optimality gap may grow, but GA should retain significant computational advantages through reduced runtime.

#### Boxplot Interpretation

The boxplot visualizations are expected to reveal:

- **Median Performance**: Indicates typical solution quality for each configuration
- **Variability Bands**: Interquartile ranges show configuration reliability; narrow bands indicate consistent performance
- **Outlier Frequency**: Rare outliers suggest robust configurations; frequent outliers indicate instability
- **Cross-Configuration Rankings**: Visual comparison identifies superior parameter combinations across the configuration space

## 9. Bibliography

Cook, W. J., Espinoza, D. G., & Goycoolea, M. (2011). Computing with cuts for capacitated vehicle routing. In *Vehicle Routing Problems: Latest Advances and New Challenges* (pp. 1–41). Springer.

Dantzig, G. B., Fulkerson, D. R., & Johnson, S. M. (1954). Solution of a large-scale traveling-salesman problem. *Journal of the Operations Research Society of America*, 2(4), 393–410.

Davis, L. (Ed.). (1991). *Handbook of Genetic Algorithms*. Van Nostrand Reinhold.

Garey, M. R., & Johnson, D. S. (1979). *Computers and Intractability: A Guide to the Theory of NP-Completeness*. W.H. Freeman and Company.

Goldberg, D. E. (1989). *Genetic Algorithms in Search, Optimization, and Machine Learning*. Addison-Wesley.

Holland, J. H. (1975). *Adaptation in Natural and Artificial Systems*. University of Michigan Press.

Karp, R. M. (1972). Reducibility among combinatorial problems. In R. E. Miller & J. W. Thatcher (Eds.), *Complexity of Computer Computations* (pp. 85–103). Plenum Press.

Larranaga, P., Kuijpers, C. M. H., Murga, R. H., Inza, I., & Dizdarevic, S. (1999). Genetic algorithms for the travelling salesman problem: A review of representations and operators. *Journal of Artificial Intelligence Research*, 13, 129–170.

Mitchell, M. (1998). *An Introduction to Genetic Algorithms*. MIT Press.

Pecin, D., Pessoa, A., Poggi, M., & Uchoa, E. (2017). Improved branch-cut-and-price for capacitated vehicle routing. *Mathematical Programming Computation*, 9(1), 61–100.

Potvin, J.-Y. (1996). Genetic algorithms for the traveling salesman problem. *Annals of Operations Research*, 63(3), 339–370.

Solomon, M. R. (1987). Algorithms for the vehicle routing and scheduling problem with time window constraints. *Operations Research*, 35(2), 254–265.

Toth, P., & Vigo, D. (Eds.). (2014). *Vehicle Routing Problems: Latest Advances and New Challenges*. Springer.

Vignaux, G. A., & Michalewicz, Z. (1991). A genetic algorithm for the linear transportation problem. *IEEE Transactions on Systems, Man, and Cybernetics*, 21(2), 445–452.

Whitley, D., Starkweather, T., & Fuquay, D. (1989). Scheduling problems and traveling salesman: The genetic edge recombination operator. In *Proceedings of the Third International Conference on Genetic Algorithms* (pp. 133–140). Morgan Kaufmann.

## 10. Conclusion

This deliverable establishes the mathematical and algorithmic foundation for optimizing ADEME's last-mile delivery operations. The problem has been formally characterized as a single-vehicle Traveling Salesman Problem with a fixed depot, complete knowledge of travel costs, and no temporal or capacity constraints.

The **Genetic Algorithm** is selected as the primary solution method, balancing computational tractability against solution quality. This choice is well-justified by established literature on GA applications to routing problems and aligns with the project's pedagogical objectives in advanced algorithms. The GA framework provides a flexible foundation for accommodating future problem extensions and operational refinements.

The theoretical study establishes the NP-hardness of the TSP, justifying the adoption of metaheuristic rather than exact algorithmic approaches for realistic problem instances. Comprehensive scientific references provide both theoretical grounding and practical guidance for algorithm design and implementation.

Subsequent deliverables will focus on algorithm implementation, empirical performance evaluation, and integration with real-world ADEME data to validate the modeling framework and demonstrate practical applicability.